**Pro**
1. Используем фильм с большим количеством отзывов на сайте Кинопоиск.
Возьмем фильм «Звездные войны: Последние джедаи», т.к. у него больше 100 и плохих, и хороших отзывов.
https://www.kinopoisk.ru/film/718223/


2. Сформируйте 3 отдельных датасета:
  - по 10 хороших и плохих отзывов
  - по 50 хороших и плохих отзывов
  - по 100 хороших и плохих отзывов

3. Сформируйте обучающие наборы данных

4. Подберите архитектуру нейронной сети для классификации собранной базы

5. Создайте таблицу с результатами

In [ ]:
import gdown # загрузка данных с Gdrive
import pickle # сериализация объектов (сохранение переменных в файлы)
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, recall_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM, Conv1D, GlobalMaxPooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# скачиваем датасет по ссылке
dataset_url = 'https://storage.yandexcloud.net/aiueducation/Content/DS/L8/dataset.pickle'
gdown.download(dataset_url, quiet=True)

'dataset.pickle'

In [ ]:
# загружаем датасет в переменные со списками
with open('dataset.pickle','rb') as f:
    all_texts_good, all_texts_bad = pickle.load(f)

In [ ]:
print(f'ХОРОШИЕ ОТЗЫВЫ ({len(all_texts_good)} штук):\n')
for t in all_texts_good[:3]:
    print(t)
    print('*'*100)

ХОРОШИЕ ОТЗЫВЫ (158 штук):

В целом, фильм вполне гармонично вписался в ряд с остальными. На наш взгляд, чтобы преисполниться и все стало абсолютно понятно, рекомендуем смотреть с первого эпизода по порядку.
****************************************************************************************************
Я бы даже еще раз сходил, очень хорошо. Потрясающее зрелище.
****************************************************************************************************
6 из 10. Снимаю балы за пресную игру престарелых актёров, утомляющий сценарий, поднадоевшую толерантность и явный коммерческий расчёт. Хотя в целом восприятие фильма позитивное. Вот оно, продолжение которое мы ждали все эти годы. Именно восьмой фильм саги для меня приблизился к планке оригинальной трилогии.
****************************************************************************************************


In [ ]:
print(f'ПЛОХИЕ ОТЗЫВЫ ({len(all_texts_bad)} штук):\n')
for t in all_texts_bad[:3]:
    print(t)
    print('*'*100)

ПЛОХИЕ ОТЗЫВЫ (167 штук):

При неплохой технической стороне и достойной игре актеров из старой гвардии – прекрасной Керри Фишер и не растерявшего свое мастерство Марка Хэмилла, а также более складного образа персонажа Драйвера Кайло Рена, получившийся итог – новое разочарование, которое сыграло на чувстве ностальгии по оригинальной трилогии, вновь выдрало оттуда целые сценарные куски, образовав невнятное «лоскутное одеяло», но так и не стало целостной историей. Новые герои так и не смогли составить конкуренцию классическим персонажам. Довольны, пожалуй, только будут представители мерчендайзингового департамента, чей план продаж будет явно перевыполнен в ближайшие месяцы. Полностью бездушная поделка, продолжающая конвейерный слив всех традиций полюбившихся старых «Звездных войн», безусловно, окупится в прокате и найдет своих фанатов, однако, создаст много трудностей для создания 9 эпизода и пока негласно провозглашает смерть вселенной «Звездных войн» от продюсерской жадности и неумения 

In [ ]:
# Формирование датасетов
datasets = {
    'dataset_10': (all_texts_good[:10] + all_texts_bad[:10], [1] * 10 + [0] * 10),
    'dataset_50': (all_texts_good[:50] + all_texts_bad[:50], [1] * 50 + [0] * 50),
    'dataset_100': (all_texts_good[:100] + all_texts_bad[:100], [1] * 100 + [0] * 100)
}

In [ ]:
# Векторизация текста
def vectorize_texts(texts, max_features=5000, max_len=100):
    vectorizer = TfidfVectorizer(max_features=max_features)
    X = vectorizer.fit_transform(texts).toarray()
    return X

In [ ]:
# Подготовка данных для нейронных сетей
def prepare_data(texts, labels, max_len=100):
    tokenizer = Tokenizer(num_words=5000)
    tokenizer.fit_on_texts(texts)
    sequences = tokenizer.texts_to_sequences(texts)
    X = pad_sequences(sequences, maxlen=max_len)
    y = np.array(labels)
    return X, y

In [ ]:
# Модель 1: Полносвязная нейронная сеть
def build_dense_model(input_dim):
    model = Sequential([
        Dense(128, activation='relu', input_dim=input_dim),
        Dense(64, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
# Модель 2: Сверточная нейронная сеть (CNN)
def build_cnn_model(input_dim):
    model = Sequential([
        Embedding(input_dim=input_dim, output_dim=128, input_length=100),
        Conv1D(128, 5, activation='relu'),
        GlobalMaxPooling1D(),
        Dense(64, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
# Модель 3: Рекуррентная нейронная сеть (LSTM)
def build_lstm_model(input_dim):
    model = Sequential([
        Embedding(input_dim=input_dim, output_dim=128, input_length=100),
        LSTM(128),
        Dense(64, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
# Обучение и оценка моделей
results = []

for dataset_name, (texts, labels) in datasets.items():
    # Векторизация текста
    X_tfidf = vectorize_texts(texts)
    X_seq, y = prepare_data(texts, labels)

    # Разделение данных
    X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)
    X_train_seq, X_test_seq, _, _ = train_test_split(X_seq, y, test_size=0.2, random_state=42)

    # Модель 1: Полносвязная нейронная сеть
    model_dense = build_dense_model(input_dim=X_train_tfidf.shape[1])
    model_dense.fit(X_train_tfidf, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=0)
    y_pred_dense = model_dense.predict(X_test_tfidf).round()
    acc_dense = accuracy_score(y_test, y_pred_dense)
    f1_dense = f1_score(y_test, y_pred_dense)
    recall_dense = recall_score(y_test, y_pred_dense)

    # Модель 2: CNN
    model_cnn = build_cnn_model(input_dim=5000)
    model_cnn.fit(X_train_seq, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=0)
    y_pred_cnn = model_cnn.predict(X_test_seq).round()
    acc_cnn = accuracy_score(y_test, y_pred_cnn)
    f1_cnn = f1_score(y_test, y_pred_cnn)
    recall_cnn = recall_score(y_test, y_pred_cnn)

    # Модель 3: LSTM
    model_lstm = build_lstm_model(input_dim=5000)
    model_lstm.fit(X_train_seq, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=0)
    y_pred_lstm = model_lstm.predict(X_test_seq).round()
    acc_lstm = accuracy_score(y_test, y_pred_lstm)
    f1_lstm = f1_score(y_test, y_pred_lstm)
    recall_lstm = recall_score(y_test, y_pred_lstm)

    # Сохранение результатов
    results.append({
        'dataset': dataset_name,
        'model_dense': {'accuracy': acc_dense, 'f1': f1_dense, 'recall': recall_dense},
        'model_cnn': {'accuracy': acc_cnn, 'f1': f1_cnn, 'recall': recall_cnn},
        'model_lstm': {'accuracy': acc_lstm, 'f1': f1_lstm, 'recall': recall_lstm}
    })

In [ ]:
# Вывод результатов
for result in results:
    print(f"Датасет: {result['dataset']}")
    print(f"Полносвязная сеть: Accuracy={result['model_dense']['accuracy']}, F1={result['model_dense']['f1']}, Recall={result['model_dense']['recall']}")
    print(f"CNN: Accuracy={result['model_cnn']['accuracy']}, F1={result['model_cnn']['f1']}, Recall={result['model_cnn']['recall']}")
    print(f"LSTM: Accuracy={result['model_lstm']['accuracy']}, F1={result['model_lstm']['f1']}, Recall={result['model_lstm']['recall']}")
    print('-' * 50)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


...

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 225ms/step
Датасет: dataset_10
Полносвязная сеть: Accuracy=0.5, F1=0.0, Recall=0.0
CNN: Accuracy=1.0, F1=1.0, Recall=1.0
LSTM: Accuracy=0.5, F1=0.0, Recall=0.0
--------------------------------------------------
Датасет: dataset_50
Полносвязная сеть: Accuracy=0.7, F1=0.6666666666666666, Recall=0.5
CNN: Accuracy=0.45, F1=0.15384615384615385, Recall=0.08333333333333333
LSTM: Accuracy=0.5, F1=0.2857142857142857, Recall=0.16666666666666666
--------------------------------------------------
Датасет: dataset_100
Полносвязная сеть: Accuracy=0.85, F1=0.8421052631578947, Recall=0.7619047619047619
CNN: Accuracy=0.6, F1=0.4666666666666667, Recall=0.3333333333333333
LSTM: Accuracy=0.575, F1=0.45161290322580644, Recall=0.3333333333333333
--------------------------------------------------
